## Reproducibility

Run all cells sequentially from top to bottom.

The default configuration reproduces the finite-domain numerical
experiment reported in the accompanying manuscript. All generated
figures and numerical diagnostics are written to
`riemann_xi_lab_outputs/`.

Typical dependencies:
- Python >= 3.10
- NumPy
- Matplotlib
- mpmath

The full grid calculation uses arbitrary-precision arithmetic and
may require several minutes depending on the hardware.

"""
Transverse Structure of the Riemann xi-Function
================================================

Numerical experiments accompanying the manuscript:

    "Transverse Structure of the Riemann ξ-Function:
     Symmetry, Positivity, and Numerical Investigation"

Author
------
Ana Isabel Castillo Pereda

Purpose
-------
This program investigates the transverse geometry of the completed
Riemann xi-function

    xi(s) = 1/2 s(s-1) pi^(-s/2) Gamma(s/2) zeta(s),

with particular emphasis on

    U(sigma,t) = log |xi(sigma + it)|,
    F(sigma,t) = d_sigma U,
    G(sigma,t) = d_sigma^2 U,

and the symmetry-adapted diagnostic field

    H(sigma,t) = (sigma - 1/2) F(sigma,t).

The computations provide finite-domain numerical diagnostics and
falsification tests. They do not constitute a proof of the
Riemann Hypothesis or of a global positivity property for H.

Numerical methodology
---------------------
* arbitrary-precision arithmetic with mpmath;
* centered finite differences in the transverse sigma direction;
* symmetric sampling about sigma = 1/2;
* exclusion of prescribed neighborhoods of known zeros for selected
  diagnostics;
* precision and finite-difference step-size stability checks;
* high-precision re-evaluation of the smallest sampled H values.

Outputs
-------
The script generates the six numerical figures used in the manuscript,
a CSV table containing rechecked candidate minima, and a numerical
summary file.

Reproducibility
---------------
Default computational domain:

    0.05 <= sigma <= 0.95
    0 <= t <= 35

The numerical parameters used in the manuscript are explicitly defined
in the configuration section below.

Last updated: September 2026
"""

In [1]:
from __future__ import annotations

import csv
from pathlib import Path

import mpmath as mp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import TwoSlopeNorm

# ============================================================
# CONFIGURATION
# ============================================================

In [2]:
OUTDIR = Path("riemann_xi_lab_outputs")
OUTDIR.mkdir(parents=True, exist_ok=True)


# Publication output

In [3]:
DPI = 300

# Numerical precision

In [4]:
MP_DPS = 50


# Symmetric sigma grid is essential for the antisymmetry test.

In [5]:
SIGMA_MIN, SIGMA_MAX = 0.05, 0.95
T_MIN, T_MAX = 0.0, 35.0


# "Paper" grid.  Increase after the first successful run.

In [6]:
NS = 81
NT = 151

SIGMA = np.linspace(SIGMA_MIN, SIGMA_MAX, NS)
TVAL = np.linspace(T_MIN, T_MAX, NT)

# Central finite-difference step in the sigma direction.
# With mpmath high precision, this is small enough for a stable
# first exploration while avoiding catastrophic double-precision loss.

In [7]:
H_DIFF = mp.mpf("1e-5")


# Known low-lying positive ordinates of critical-line zeros.

In [8]:
ZERO_T = np.array([
    14.134725141734693,
    21.022039638771555,
    25.010857580145690,
    30.424876125859513,
    32.935061587739190,
], dtype=float)

# For diagnostic statistics we exclude a narrow vertical neighborhood
# around known zeros because log|xi| is singular at a zero.

In [9]:
ZERO_T_EXCLUSION = 0.08

# Candidate-violation threshold used only for the exported table.

In [10]:
H_VIOLATION_TOL = 1e-8

# ============================================================
# PUBLICATION STYLE
# ============================================================

In [18]:
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10.5,
    "axes.titlesize": 12.5,
    "axes.labelsize": 11.0,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.fontsize": 9.2,
    "figure.titlesize": 13.5,
    "axes.linewidth": 0.8,
    "savefig.bbox": "tight",
})

GOLD = "#B58B2A"
BLUE = "#235789"
RED = "#C44536"
DARK = "#18212B"
GRID = "#D8DDE3"

# ============================================================
# MATHEMATICAL CORE
# ============================================================

In [19]:
mp.mp.dps = MP_DPS

def xi(s: mp.mpc) -> mp.mpc:
    """Completed Riemann xi-function."""
    return (
        mp.mpf("0.5")
        * s * (s - 1)
        * mp.power(mp.pi, -s / 2)
        * mp.gamma(s / 2)
        * mp.zeta(s)
    )

def logabs_xi(sigma: float, t: float) -> mp.mpf:
    """Natural logarithm of |xi(sigma+it)|."""
    s = mp.mpc(sigma, t)
    z = xi(s)
    return mp.log(abs(z))

def transverse_quantities(
    sigma: float,
    t: float,
    h: mp.mpf = H_DIFF,
) -> tuple[float, float, float]:
    """
    Return (L,F,G) where
        L = log|xi|,
        F = d_sigma L,
        G = d_sigma^2 L,
    using symmetric high-precision differences.
    """
    sm = mp.mpf(str(sigma)) - h
    s0 = mp.mpf(str(sigma))
    sp = mp.mpf(str(sigma)) + h
    tt = mp.mpf(str(t))

    lm = logabs_xi(sm, tt)
    l0 = logabs_xi(s0, tt)
    lp = logabs_xi(sp, tt)

    F = (lp - lm) / (2 * h)
    G = (lp - 2*l0 + lm) / (h*h)

    return float(l0), float(F), float(G)

def near_known_zero(t: float, width: float = ZERO_T_EXCLUSION) -> bool:
    return bool(np.any(np.abs(ZERO_T - t) < width))


# ============================================================
# COMPUTE GRID
# ============================================================

In [20]:
print(f"Computing xi-grid: NS={NS}, NT={NT}, mp.dps={MP_DPS}")
print("This is a publication calculation, not the fast cinematic approximation.")

LOGABS = np.empty((NT, NS), dtype=float)
F = np.empty((NT, NS), dtype=float)
G = np.empty((NT, NS), dtype=float)

for j, t in enumerate(TVAL):
    if j % max(1, NT // 10) == 0:
        print(f"  row {j+1:4d}/{NT}   t={t:8.4f}")

    for i, sigma in enumerate(SIGMA):
        try:
            l0, f0, g0 = transverse_quantities(float(sigma), float(t))
            LOGABS[j, i] = l0
            F[j, i] = f0
            G[j, i] = g0
        except (ValueError, ZeroDivisionError, OverflowError):
            LOGABS[j, i] = np.nan
            F[j, i] = np.nan
            G[j, i] = np.nan

# Convert ln|xi| to log10|xi| for intuitive plotting.
LOG10ABS = LOGABS / np.log(10.0)

# Symmetry-adapted diagnostic field.
HFIELD = (SIGMA[None, :] - 0.5) * F

# Exact grid mirror because SIGMA is symmetric around 1/2.
F_MIRROR = F[:, ::-1]
ANTI_RESID = F + F_MIRROR

# Exclusion mask for diagnostics near known singular zeros.
DIAG_MASK = np.isfinite(F)
for j, t in enumerate(TVAL):
    if near_known_zero(float(t)):
        DIAG_MASK[j, :] = False
# Exclude the critical-line column when evaluating the smallest
# sampled off-axis value of H. Since H(1/2,t)=0 identically,
# including sigma=1/2 would make the minimum trivially zero.
OFF_AXIS_MASK = DIAG_MASK.copy()

i_half = int(np.argmin(np.abs(SIGMA - 0.5)))
OFF_AXIS_MASK[:, i_half] = False

Computing xi-grid: NS=81, NT=151, mp.dps=50
This is a publication calculation, not the fast cinematic approximation.
  row    1/151   t=  0.0000
  row   16/151   t=  3.5000
  row   31/151   t=  7.0000
  row   46/151   t= 10.5000
  row   61/151   t= 14.0000
  row   76/151   t= 17.5000
  row   91/151   t= 21.0000
  row  106/151   t= 24.5000
  row  121/151   t= 28.0000
  row  136/151   t= 31.5000
  row  151/151   t= 35.0000


# ============================================================
# NUMERICAL DIAGNOSTICS
# ============================================================

In [22]:
anti_vals = np.abs(ANTI_RESID[DIAG_MASK])
anti_max = float(np.nanmax(anti_vals)) if anti_vals.size else np.nan
anti_rms = float(np.sqrt(np.nanmean(anti_vals**2))) if anti_vals.size else np.nan

h_valid = HFIELD[OFF_AXIS_MASK]
h_min = float(np.nanmin(h_valid)) if h_valid.size else np.nan
h_neg_count = int(np.sum(h_valid < -H_VIOLATION_TOL))

# Locate the minimum sampled H value on the admissible grid.
H_SEARCH = np.where(OFF_AXIS_MASK, HFIELD, np.nan)
flat_idx = np.nanargmin(H_SEARCH)
jmin, imin = np.unravel_index(flat_idx, H_SEARCH.shape)
sigma_min_h = float(SIGMA[imin])
t_min_h = float(TVAL[jmin])

print("\nDiagnostics")
print("-----------")
print(f"max |F(sigma,t)+F(1-sigma,t)| : {anti_max:.6e}")
print(f"RMS antisymmetry residual      : {anti_rms:.6e}")
print(f"minimum H away from zeros      : {h_min:.12e}")
print(f"grid points with H < -tol      : {h_neg_count}")
print(f"minimum located near           : sigma={sigma_min_h:.6f}, t={t_min_h:.6f}")


Diagnostics
-----------
max |F(sigma,t)+F(1-sigma,t)| : 1.435177e-14
RMS antisymmetry residual      : 2.530979e-16
minimum H away from zeros      : 5.848450191474e-06
grid points with H < -tol      : 0
minimum located near           : sigma=0.511250, t=0.000000




# ============================================================
# HIGH-PRECISION RECHECK OF MOST SUSPICIOUS H POINTS
# ============================================================

In [23]:
candidate_rows = []

# Take the 20 smallest finite H values outside zero neighborhoods.
indices = np.argwhere(OFF_AXIS_MASK & np.isfinite(HFIELD))
values = np.array([HFIELD[j, i] for j, i in indices])
order = np.argsort(values)[:20]

for rank, pos in enumerate(order, start=1):
    j, i = indices[pos]
    sigma = float(SIGMA[i])
    t = float(TVAL[j])

    row = {
        "rank": rank,
        "sigma": sigma,
        "t": t,
        "H_grid": float(HFIELD[j, i]),
        "F_grid": float(F[j, i]),
        "G_grid": float(G[j, i]),
        "log10_abs_xi": float(LOG10ABS[j, i]),
    }

    # Independent precision/step rechecks.
    for dps, hs in [(50, "1e-5"), (70, "3e-6"), (90, "1e-6")]:
        with mp.workdps(dps):
            _, fr, gr = transverse_quantities(sigma, t, mp.mpf(hs))
            hr = (sigma - 0.5) * fr
        row[f"H_dps{dps}"] = hr
        row[f"F_dps{dps}"] = fr
        row[f"G_dps{dps}"] = gr

    candidate_rows.append(row)

csv_path = OUTDIR / "candidate_H_minima_rechecked.csv"
with csv_path.open("w", newline="", encoding="utf-8") as fcsv:
    writer = csv.DictWriter(fcsv, fieldnames=list(candidate_rows[0].keys()))
    writer.writeheader()
    writer.writerows(candidate_rows)

# ============================================================
# HELPERS FOR FIGURES
# ============================================================

In [24]:
extent = [SIGMA_MIN, SIGMA_MAX, T_MIN, T_MAX]

def add_critical_line(ax):
    ax.axvline(0.5, color=GOLD, lw=1.8, ls="--",
               label=r"critical line $\sigma=\frac{1}{2}$")

def add_known_zeros(ax):
    z = ZERO_T[(ZERO_T >= T_MIN) & (ZERO_T <= T_MAX)]
    ax.scatter(np.full_like(z, 0.5), z, s=28,
               facecolors="white", edgecolors=GOLD,
               linewidths=1.2, zorder=7,
               label="known low-lying zeros")

def save_pub(fig, stem):
    png = OUTDIR / f"{stem}.png"
    pdf = OUTDIR / f"{stem}.pdf"
    fig.savefig(png, dpi=DPI, facecolor="white")
    fig.savefig(pdf, facecolor="white")
    plt.close(fig)
    print("saved:", png)
    print("saved:", pdf)



# ============================================================
# FIGURE 1 - xi landscape
# ============================================================

In [32]:
fig, ax = plt.subplots(figsize=(7.4, 5.0))
v1, v99 = np.nanpercentile(LOG10ABS, [2, 98])
im = ax.imshow(
    LOG10ABS, origin="lower", aspect="auto", extent=extent,
    cmap="viridis", vmin=v1, vmax=v99
)
add_critical_line(ax)
add_known_zeros(ax)
ax.set_xlabel(r"$\sigma=\mathrm{Re}(s)$")
ax.set_ylabel(r"$t=\mathrm{Im}(s)$")
ax.set_title(r"Logarithmic modulus of the completed Riemann $\xi$-function")
cb = fig.colorbar(im, ax=ax, pad=0.02)
cb.set_label(r"$\log_{10}|\xi|$")
ax.legend(loc="upper right", frameon=True)
fig.tight_layout()
save_pub(fig, "Figure01_xi_logabs_landscape")

saved: riemann_xi_lab_outputs/Figure01_xi_logabs_landscape.png
saved: riemann_xi_lab_outputs/Figure01_xi_logabs_landscape.pdf


# ============================================================
# FIGURE 2 - F antisymmetric field
# ============================================================

In [26]:
fig, ax = plt.subplots(figsize=(7.4, 5.0))
finite_F = F[np.isfinite(F)]
limF = np.nanpercentile(np.abs(finite_F), 96)
normF = TwoSlopeNorm(vmin=-limF, vcenter=0.0, vmax=limF)
im = ax.imshow(
    F, origin="lower", aspect="auto", extent=extent,
    cmap="RdBu_r", norm=normF
)
add_critical_line(ax)
add_known_zeros(ax)
ax.set_xlabel(r"$\sigma$")
ax.set_ylabel(r"$t$")
ax.set_title(
    r"Transverse field $F(\sigma,t)=\partial_\sigma\log|\xi(\sigma+it)|$"
)
cb = fig.colorbar(im, ax=ax, pad=0.02)
cb.set_label(r"$F(\sigma,t)$")
ax.text(
    0.02, 0.02,
    rf"max antisymmetry residual = {anti_max:.2e}",
    transform=ax.transAxes,
    ha="left", va="bottom",
    bbox=dict(boxstyle="round,pad=0.28", fc="white", ec="0.75", alpha=0.92)
)
fig.tight_layout()
save_pub(fig, "Figure02_transverse_F_antisymmetry")

saved: riemann_xi_lab_outputs/Figure02_transverse_F_antisymmetry.png
saved: riemann_xi_lab_outputs/Figure02_transverse_F_antisymmetry.pdf



# ============================================================
# FIGURE 3 - symmetry-adapted diagnostic field H
# ============================================================

In [27]:
fig, ax = plt.subplots(figsize=(7.4, 5.0))
finite_H = HFIELD[np.isfinite(HFIELD)]
limH = np.nanpercentile(np.abs(finite_H), 97)
normH = TwoSlopeNorm(vmin=-limH, vcenter=0.0, vmax=limH)
im = ax.imshow(
    HFIELD, origin="lower", aspect="auto", extent=extent,
    cmap="RdBu_r", norm=normH
)
add_critical_line(ax)
add_known_zeros(ax)
ax.scatter(
    [sigma_min_h], [t_min_h],
    marker="x", s=55, linewidths=1.5, color=DARK,
    label="minimum sampled $H$"
)
ax.set_xlabel(r"$\sigma$")
ax.set_ylabel(r"$t$")
ax.set_title(
    r"Falsification map: $H=(\sigma-\frac{1}{2})F(\sigma,t)$"
)
cb = fig.colorbar(im, ax=ax, pad=0.02)
cb.set_label(r"$H(\sigma,t)$")
ax.legend(loc="upper right", frameon=True)
ax.text(
    0.02, 0.02,
    rf"$H_{{\min}}={h_min:.3e}$; negative count={h_neg_count}",
    transform=ax.transAxes,
    ha="left", va="bottom",
    bbox=dict(boxstyle="round,pad=0.28", fc="white", ec="0.75", alpha=0.92)
)
fig.tight_layout()
save_pub(fig, "Figure03_H_falsification_map")

saved: riemann_xi_lab_outputs/Figure03_candidate_H_falsification.png
saved: riemann_xi_lab_outputs/Figure03_candidate_H_falsification.pdf


# ============================================================
# FIGURE 4 - G curvature field
# ============================================================

In [28]:
fig, ax = plt.subplots(figsize=(7.4, 5.0))
finite_G = G[np.isfinite(G)]
limG = np.nanpercentile(np.abs(finite_G), 96)
normG = TwoSlopeNorm(vmin=-limG, vcenter=0.0, vmax=limG)
im = ax.imshow(
    G, origin="lower", aspect="auto", extent=extent,
    cmap="PuOr_r", norm=normG
)
add_critical_line(ax)
add_known_zeros(ax)
ax.set_xlabel(r"$\sigma$")
ax.set_ylabel(r"$t$")
ax.set_title(
    r"Transverse curvature $G(\sigma,t)=\partial_\sigma^2\log|\xi(\sigma+it)|$"
)
cb = fig.colorbar(im, ax=ax, pad=0.02)
cb.set_label(r"$G(\sigma,t)$")
fig.tight_layout()
save_pub(fig, "Figure04_transverse_G_curvature")

saved: riemann_xi_lab_outputs/Figure04_transverse_G_curvature.png
saved: riemann_xi_lab_outputs/Figure04_transverse_G_curvature.pdf




# ============================================================
# FIGURE 5 - Centered transverse profiles of the xi-function
# ============================================================

In [29]:
cut_t = [8.0, 13.7, 14.55, 18.0, 20.6, 21.45, 27.0, 33.5]

fig, ax = plt.subplots(figsize=(7.2, 4.9))

# The symmetric sigma grid contains sigma = 1/2 exactly.
i_half = int(np.argmin(np.abs(SIGMA - 0.5)))

for tc in cut_t:
    j = int(np.argmin(np.abs(TVAL - tc)))

    # Centered transverse profile:
    # ΔL(σ,t) = log10|ξ(σ+it)| - log10|ξ(1/2+it)|.
    y = LOG10ABS[j, :] - LOG10ABS[j, i_half]

    ax.plot(
        SIGMA,
        y,
        lw=1.4,
        label=rf"$t \approx {TVAL[j]:.2f}$"
    )

# Critical line
ax.axvline(
    0.5,
    color=GOLD,
    lw=1.6,
    ls="--",
    label=r"critical line $\sigma=\frac{1}{2}$"
)

# Reference level imposed by centering
ax.axhline(
    0.0,
    color="0.5",
    lw=0.8,
    ls=":"
)

ax.set_xlabel(r"$\sigma$")
ax.set_ylabel(r"$\Delta L(\sigma,t)$")

ax.set_title(
    r"Centered transverse profiles of the $\xi$-function"
)

ax.grid(
    True,
    which="major",
    alpha=0.20,
    linewidth=0.7
)

ax.legend(
    ncol=2,
    loc="upper center",
    frameon=True,
    fancybox=False,
    edgecolor="0.7"
)

fig.tight_layout()

save_pub(
    fig,
    "Figure05_centered_transverse_xi_profiles"
)

saved: riemann_xi_lab_outputs/Figure05_centered_transverse_xi_profiles.png
saved: riemann_xi_lab_outputs/Figure05_centered_transverse_xi_profiles.pdf


# ============================================================
# FIGURE 6 - joint precision / step-size stability of H
# ============================================================

In [30]:
# Representative off-axis points, including the sampled point
# at which the minimum admissible H value was recorded.
validation_points = [
    (0.30, 10.0),
    (0.40, 18.0),
    (0.60, 18.0),
    (0.70, 27.0),
    (sigma_min_h, t_min_h),
]

dps_values = [30, 40, 50, 60, 70, 80]

# Precision and finite-difference step are refined jointly.
h_values = {
    30: "3e-4",
    40: "1e-4",
    50: "3e-5",
    60: "1e-5",
    70: "3e-6",
    80: "1e-6",
}

markers = ["o", "s", "^", "o", "o"]
linestyles = ["-", "--", "-.", ":", "-"]

fig, ax = plt.subplots(figsize=(7.4, 5.0))

for k, (sigma, t) in enumerate(validation_points):

    vals = []

    for dps in dps_values:
        with mp.workdps(dps):
            _, fr, _ = transverse_quantities(
                mp.mpf(str(sigma)),
                mp.mpf(str(t)),
                mp.mpf(h_values[dps])
            )

            H_val = (mp.mpf(str(sigma)) - mp.mpf("0.5")) * fr
            vals.append(H_val)

    # Numerical reference: evaluation at the largest working precision
    # and smallest finite-difference step used in this experiment.
    # This is a convergence reference, not an exact analytical value.
    ref = vals[-1]

    err = np.array(
        [float(abs(v - ref)) for v in vals],
        dtype=float
    )

    # The reference point has zero self-error. A plotting floor is
    # used solely because zero cannot be displayed on a logarithmic axis.
    err_plot = np.maximum(err, 1e-18)

    ax.semilogy(
        dps_values,
        err_plot,
        marker=markers[k],
        linestyle=linestyles[k],
        ms=4.5,
        lw=1.15,
        label=rf"$({sigma:.3f},\,{t:.3f})$"
    )

ax.set_xlabel("mpmath decimal precision (dps)")
ax.set_ylabel(r"$|H_{\mathrm{dps},h}-H_{\mathrm{ref}}|$")
ax.set_title(r"Joint precision and step-size stability of $H$")

ax.grid(
    True,
    which="both",
    alpha=0.22,
    linewidth=0.7
)

ax.legend(
    title=r"$(\sigma,t)$",
    frameon=True,
    fancybox=False,
    edgecolor="0.7"
)

fig.tight_layout()

save_pub(
    fig,
    "Figure06_precision_stepsize_stability_H_300dpi"
)

saved: riemann_xi_lab_outputs/Figure06_precision_stepsize_stability_H_300dpi.png
saved: riemann_xi_lab_outputs/Figure06_precision_stepsize_stability_H_300dpi.pdf



# ============================================================
# SUMMARY FILE
# ============================================================

In [31]:
summary_path = OUTDIR / "numerical_summary.txt"
summary_path.write_text(
    "\n".join([
        "RIEMANN XI ANALYTICAL LAB — numerical summary",
        f"mp.dps = {MP_DPS}",
        f"grid = {NS} x {NT}",
        f"sigma range = [{SIGMA_MIN}, {SIGMA_MAX}]",
        f"t range = [{T_MIN}, {T_MAX}]",
        f"finite-difference h = {H_DIFF}",
        "",
        f"max antisymmetry residual = {anti_max:.16e}",
        f"RMS antisymmetry residual = {anti_rms:.16e}",
        f"minimum H away from known-zero exclusion bands = {h_min:.16e}",
        f"negative H count (< -{H_VIOLATION_TOL:g}) = {h_neg_count}",
        f"minimum H location sigma = {sigma_min_h:.12f}",
        f"minimum H location t = {t_min_h:.12f}",
        "",
        "Interpretation:",
        "These values are numerical diagnostics only.  They do not prove RH.",
        "Any apparent sign law must survive independent precision, method,",
        "domain, and literature checks before it can be promoted to a lemma.",
    ]),
    encoding="utf-8"
)

print("\nDone.")
print("Output directory:", OUTDIR.resolve())
print("Candidate table:", csv_path)
print("Summary:", summary_path)


Done.
Output directory: /content/riemann_xi_lab_outputs
Candidate table: riemann_xi_lab_outputs/candidate_H_minima_rechecked.csv
Summary: riemann_xi_lab_outputs/numerical_summary.txt
